# CXR Imaging Analysis — MedGemma Multimodal Pipeline

**Evaluating MedGemma's 3-stage CXR analysis on 12 real RSNA chest X-ray images.**

> This notebook is generated by `_generate_cxr_analysis_notebook.py`. Do not edit manually.

### 3-Stage CXR Pipeline
1. **Classification (Stage A):** Detect consolidation, pleural effusion, cardiomegaly, edema, atelectasis
2. **Localization (Stage B):** Bounding box coordinates for detected findings (normalized 0-1000)
3. **Longitudinal (Stage C):** Compare current vs prior CXR when available

### 12 RSNA Images Across 5 Categories
| Category | Cases | What it tests |
|----------|-------|--------------|
| Clear Pneumonia | 3 | Consolidation detection + localization accuracy |
| Bilateral Pneumonia | 2 | Multi-region detection, CR-5 (laterality mismatch) |
| Subtle Pneumonia | 2 | Sensitivity on borderline findings |
| Normal CXR | 3 | True negatives, CR-1/CR-2 (clinical-radiological discordance) |
| Abnormal Non-Pneumonia | 2 | Effusion detection, CR-6 (unexpected CXR findings) |

### Quick Start
1. Set runtime to **GPU -> A100** (Runtime -> Change runtime type)
2. Add **HF_TOKEN** to Colab Secrets (key icon in left sidebar)
3. **Run All** (Runtime -> Run all)


> **Disclaimer:** This notebook demonstrates a research prototype. All clinical outputs
> (severity scores, antibiotic recommendations, contradiction alerts, CXR analysis)
> are **AI-generated by MedGemma 1.5 4B** and have been validated only on synthetic data.
> This system is **not a medical device**, is not approved for clinical use, and must not
> be used for patient care decisions. See [DISCLAIMER.md](../DISCLAIMER.md) for full terms.


In [ ]:
# Cell 1: Install package + dependencies
import os

# Detect Colab vs local
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Install from GitHub (requires GITHUB_TOKEN in Colab Secrets)
    try:
        github_token = userdata.get("GITHUB_TOKEN")
        repo_url = f"git+https://{github_token}@github.com/HP-00/MedGemma-Pneumonia-Agent.git@main"
    except Exception:
        repo_url = "git+https://github.com/HP-00/MedGemma-Pneumonia-Agent.git@main"

    %pip install --quiet {repo_url}
    %pip install --quiet plotly>=5.0.0 nest-asyncio
else:
    print("Local environment detected. Ensure: pip install -e '.[dev,benchmark]'")

import nest_asyncio
nest_asyncio.apply()

print("Setup complete")


## Authentication


In [ ]:
import os

# HuggingFace token for gated model access
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except ImportError:
    pass

assert os.environ.get("HF_TOKEN"), "HF_TOKEN not set! Add it to Colab Secrets or environment."

print("Authentication complete")


## Locate RSNA Images

The 12 RSNA CXR images are bundled with the installed package.


In [ ]:
import os, glob

# Locate RSNA images from installed package
import benchmark_data.rsna
RSNA_IMG_DIR = os.path.join(os.path.dirname(benchmark_data.rsna.__file__), "images")
rsna_images = sorted(glob.glob(os.path.join(RSNA_IMG_DIR, "*.png")))
print(f"RSNA CXR: {len(rsna_images)} images in {RSNA_IMG_DIR}")
for img in rsna_images:
    print(f"  {os.path.basename(img)}")


## Load Model & Build Graph


In [ ]:
import time
from cap_agent.models.medgemma import get_model_and_processor
from cap_agent.agent.graph import build_cap_agent_graph

# Load MedGemma (lazy singleton - only loads once, includes warm-up)
model, processor = get_model_and_processor()
print(f"Model loaded on {model.device}")

# Build the 8-node LangGraph pipeline
graph = build_cap_agent_graph()
print("Graph compiled: 8 nodes, conditional routing at contradiction check")


## CXR Pipeline Overview

The CXR extraction module (`cap_agent.data.extraction`) runs up to 3 MedGemma GPU calls per image:

### Stage A: Classification
Detects 5 findings: consolidation, pleural effusion, cardiomegaly, edema, atelectasis.
Each finding gets a present/absent flag with confidence level (high/moderate/low).
Also assesses image quality (projection, rotation, penetration).

### Stage B: Localization
For each finding detected as present, generates a bounding box in normalized coordinates
(0-1000 scale). Only runs if Stage A detects at least one finding.

### Stage C: Longitudinal Comparison
When a prior CXR is available, compares current vs prior for each finding category.
Reports change direction (improved/worsened/new/resolved/stable) with description.

**GPU calls per case:** 1 (classification only) to 3 (classification + localization + longitudinal).


In [ ]:
from PIL import Image, ImageDraw, ImageFont
import IPython.display

def display_cxr_with_bbox(image_path, predicted_bboxes, gt_bboxes, title="CXR"):
    """Display a CXR image with predicted (red) and ground truth (green) bounding boxes.

    Args:
        image_path: Path to the CXR image file.
        predicted_bboxes: List of {"bbox": [y0, x0, y1, x1], "label": str} (normalized 0-1000).
        gt_bboxes: List of {"bbox": [y0, x0, y1, x1], "label": str} (normalized 0-1000).
        title: Display title for the image.
    """
    img = Image.open(image_path).copy()
    orig_w, orig_h = img.size

    # Resize for display
    display_w = 512
    display_h = int(orig_h * (display_w / orig_w))
    img = img.resize((display_w, display_h))
    draw = ImageDraw.Draw(img)

    # Draw ground truth boxes (green)
    for item in gt_bboxes:
        bbox = item["bbox"]
        if len(bbox) == 4:
            y0, x0, y1, x1 = bbox
            px0 = x0 / 1000 * display_w
            py0 = y0 / 1000 * display_h
            px1 = x1 / 1000 * display_w
            py1 = y1 / 1000 * display_h
            draw.rectangle([(px0, py0), (px1, py1)], outline="green", width=2)
            draw.text((px0, max(py0 - 12, 0)), f"GT", fill="green")

    # Draw predicted boxes (red)
    for item in predicted_bboxes:
        bbox = item["bbox"]
        label = item.get("label", "pred")
        if len(bbox) == 4:
            y0, x0, y1, x1 = bbox
            px0 = x0 / 1000 * display_w
            py0 = y0 / 1000 * display_h
            px1 = x1 / 1000 * display_w
            py1 = y1 / 1000 * display_h
            draw.rectangle([(px0, py0), (px1, py1)], outline="red", width=2)
            draw.text((px0, max(py0 - 12, 0)), label, fill="red")

    print(f"\n--- {title} ---")
    print(f"  Predicted boxes (red): {len(predicted_bboxes)}")
    print(f"  Ground truth boxes (green): {len(gt_bboxes)}")
    display(img)


def compute_iou(pred_bbox, gt_bbox):
    """Compute Intersection-over-Union between two bounding boxes.

    Both boxes are [y0, x0, y1, x1] in normalized 0-1000 coordinates.
    Returns float in [0.0, 1.0].
    """
    py0, px0, py1, px1 = pred_bbox
    gy0, gx0, gy1, gx1 = gt_bbox

    inter_x0 = max(px0, gx0)
    inter_y0 = max(py0, gy0)
    inter_x1 = min(px1, gx1)
    inter_y1 = min(py1, gy1)

    inter_w = max(0, inter_x1 - inter_x0)
    inter_h = max(0, inter_y1 - inter_y0)
    inter_area = inter_w * inter_h

    pred_area = (px1 - px0) * (py1 - py0)
    gt_area = (gx1 - gx0) * (gy1 - gy0)
    union_area = pred_area + gt_area - inter_area

    if union_area <= 0:
        return 0.0
    return inter_area / union_area

print("display_cxr_with_bbox() and compute_iou() defined")


In [ ]:
# Clinical output renderer — formats structured_output as a readable document
def render_clinical_output(result, scenario_title="Pipeline Output"):
    """Render structured_output as a formatted clinical document using ANSI codes."""
    ESC = chr(27)
    B = f"{ESC}[1m"       # Bold
    R = f"{ESC}[91m"      # Red
    Y = f"{ESC}[93m"      # Yellow
    G = f"{ESC}[92m"      # Green
    C = f"{ESC}[96m"      # Cyan
    X = f"{ESC}[0m"       # Reset

    so = result.get("structured_output", {})
    sections = so.get("sections", {})

    print(f"\n{B}{C}{'=' * 70}")
    print(f"  CLINICAL DECISION SUPPORT — {scenario_title}")
    print(f"{'=' * 70}{X}")
    print(f"{Y}AI-generated observations for clinician review — not a substitute for clinical judgement.{X}\n")

    # 1. PATIENT
    s1 = sections.get("1_patient", {})
    demo = s1.get("demographics", {})
    print(f"{B}{C}1. PATIENT{X}")
    print(f"  {demo.get('age', '?')}yo {demo.get('sex', '?')}")
    allergy_list = demo.get("allergies", [])
    if allergy_list:
        for a in allergy_list:
            if isinstance(a, dict):
                print(f"  {R}Allergy: {a.get('drug', '?')} — {a.get('reaction_type', '?')} ({a.get('severity', '?')}){X}")
            else:
                print(f"  {R}Allergy: {a}{X}")
    else:
        print(f"  {G}No known drug allergies{X}")
    combos = demo.get("comorbidities", [])
    if combos:
        print(f"  Comorbidities: {', '.join(combos)}")
    if demo.get("pregnancy"):
        print(f"  {R}PREGNANT{X}")
    print(f"  Oral tolerance: {'Yes' if demo.get('oral_tolerance', True) else R + 'No' + X}")
    travel = demo.get("travel_history", [])
    if travel:
        print(f"  {Y}Travel: {', '.join(travel)}{X}")
    print()

    # 2. SEVERITY
    s2 = sections.get("2_severity", {})
    curb = s2.get("curb65", {})
    sev = curb.get("severity_tier", "?")
    sev_color = R if sev == "high" else (Y if sev == "moderate" else G)
    score = curb.get("curb65")
    score_str = f"CURB65={score}" if score is not None else f"CRB65={curb.get('crb65', '?')} (urea unavailable)"
    print(f"{B}{C}2. SEVERITY{X}")
    print(f"  {score_str}  {sev_color}{B}{sev.upper()}{X}")
    print(f"  C={curb.get('c','?')} U={curb.get('u','?')} R={curb.get('r','?')} B={curb.get('b','?')} 65={curb.get('age_65','?')}")
    missing = curb.get("missing_variables", [])
    if missing:
        print(f"  {Y}Missing: {', '.join(missing)}{X}")
    poc = s2.get("place_of_care", {})
    if poc:
        print(f"  Place of care: {poc.get('recommendation', '?')}")
    print()

    # 3. CXR
    s3 = sections.get("3_cxr", {})
    cxr = s3.get("findings", {})
    print(f"{B}{C}3. CHEST X-RAY{X}")
    for finding in ["consolidation", "pleural_effusion", "cardiomegaly", "edema", "atelectasis"]:
        f = cxr.get(finding, {})
        if not isinstance(f, dict):
            continue
        present = f.get("present", False)
        conf = f.get("confidence", "?")
        if present:
            loc = f.get("location", "")
            bbox = f.get("bounding_box")
            line = f"  {R}PRESENT{X} {finding} ({conf} confidence)"
            if loc:
                line += f" at {loc}"
            if bbox:
                line += f" [bbox: {bbox}]"
            print(line)
        else:
            print(f"  {G}absent{X}  {finding} ({conf} confidence)")
    iq = cxr.get("image_quality", {})
    if iq:
        print(f"  Quality: {iq.get('projection','?')} / {iq.get('rotation','?')} / {iq.get('penetration','?')}")
    longit = cxr.get("longitudinal_comparison")
    if longit:
        print(f"  {B}Longitudinal:{X}")
        for fn, ch in longit.items():
            if isinstance(ch, dict):
                print(f"    {fn}: {ch.get('change','?')} — {ch.get('description','')}")
    print()

    # 4. KEY BLOODS
    s4 = sections.get("4_key_bloods", {})
    labs = s4.get("values", {})
    print(f"{B}{C}4. KEY BLOODS{X}")
    for test, data in (labs or {}).items():
        if not isinstance(data, dict):
            continue
        val = data.get("value", "?")
        unit = data.get("unit", "")
        ref = data.get("reference_range", "")
        abn = data.get("abnormal_flag", False)
        color = R if abn else G
        flag = " ABNORMAL" if abn else ""
        print(f"  {color}{test}: {val} {unit}{flag}{X}  (ref: {ref})")
    print()

    # 5. CONTRADICTIONS
    s5 = sections.get("5_contradiction_alert", {})
    alerts = s5.get("alerts", [])
    informational = s5.get("informational", [])
    resolutions = s5.get("resolutions", [])
    n_total = s5.get("detected", 0)
    print(f"{B}{C}5. CONTRADICTION ALERTS{X}")
    if n_total == 0:
        print(f"  {G}No contradictions detected — findings concordant{X}")
    else:
        print(f"  {n_total} contradiction(s) detected")
        for c in alerts:
            conf = c.get("confidence", "?")
            badge_color = R if conf == "high" else Y
            print(f"  {badge_color}[{conf.upper()}]{X} {c.get('rule_id','?')}: {c.get('pattern','')}")
            if c.get("evidence_for"):
                print(f"    For: {c['evidence_for']}")
            if c.get("evidence_against"):
                print(f"    Against: {c['evidence_against']}")
            rec = c.get("recommendation")
            if rec:
                print(f"    {B}Recommendation:{X} {rec}")
        for c in informational:
            print(f"  {G}[LOW]{X} {c.get('rule_id','?')}: {c.get('pattern','')}")
        if resolutions:
            print(f"  {B}Resolutions:{X}")
            for r in resolutions:
                print(f"    {r[:200]}")
    print()

    # 6. TREATMENT
    s6 = sections.get("6_treatment_pathway", {})
    abx = s6.get("antibiotic", {})
    print(f"{B}{C}6. TREATMENT{X}")
    print(f"  First-line: {abx.get('first_line', 'N/A')}")
    print(f"  Dose/route: {abx.get('dose_route', 'N/A')}")
    if abx.get("allergy_adjustment"):
        print(f"  {Y}Allergy adj: {abx['allergy_adjustment']}{X}")
    if abx.get("atypical_cover"):
        print(f"  Atypical: {abx['atypical_cover']}")
    if abx.get("renal_adjustment"):
        print(f"  {Y}Renal: {abx['renal_adjustment']}{X}")
    cortico = s6.get("corticosteroid")
    if cortico:
        print(f"  Corticosteroid: {cortico}")
    steward = abx.get("stewardship_notes", [])
    for note in steward:
        print(f"  {Y}Stewardship: {note}{X}")
    inv = s6.get("investigations", {})
    if inv:
        print(f"  {B}Investigations:{X}")
        for name, detail in inv.items():
            if isinstance(detail, dict):
                rec = "Recommended" if detail.get("recommended") else "Not indicated"
                print(f"    {name}: {rec} — {detail.get('reasoning', '')[:100]}")
    print()

    # 7. DATA GAPS
    s7 = sections.get("7_data_gaps", {})
    gaps = s7.get("gaps", [])
    print(f"{B}{C}7. DATA GAPS{X}")
    if gaps:
        for g in gaps:
            print(f"  {Y}- {g}{X}")
    else:
        print(f"  {G}None identified{X}")
    print()

    # 8. MONITORING
    s8 = sections.get("8_monitoring", {})
    plan = s8.get("plan", {})
    print(f"{B}{C}8. MONITORING{X}")
    crp_trend = plan.get("crp_trend")
    if crp_trend:
        adm = crp_trend.get("admission_value", "?")
        cur = crp_trend.get("current_value", "?")
        pct = crp_trend.get("percent_change", "?")
        trend = crp_trend.get("trend", "?")
        sr = crp_trend.get("flag_senior_review", False)
        sr_color = R if sr else G
        print(f"  CRP: {adm} -> {cur} mg/L ({pct}% change, {trend})")
        print(f"  Senior review: {sr_color}{sr}{X}")
    tr = plan.get("treatment_response")
    if tr:
        reassess = tr.get("reassess_needed", False)
        ra_color = R if reassess else G
        print(f"  Treatment response: {ra_color}reassess_needed={reassess}{X}")
        for action in tr.get("actions", []):
            print(f"    - {action}")
    dc = plan.get("discharge_criteria_met")
    if dc is not None:
        dc_color = G if dc else R
        dc_str = "MET" if dc else "NOT MET"
        print(f"  Discharge: {dc_color}{dc_str}{X}")
    dc_detail = plan.get("discharge_criteria_details", {})
    if dc_detail:
        for check, passed_val in dc_detail.items():
            if check.startswith("_"):
                continue
            chk_color = G if passed_val else R
            chk_str = "PASS" if passed_val else "FAIL"
            print(f"    {chk_color}[{chk_str}]{X} {check}")
    print(f"  Next review: {plan.get('next_review', 'N/A')}")
    print()

    # PROVENANCE
    prov = so.get("provenance", {})
    if prov:
        print(f"{B}{C}PROVENANCE{X}")
        tools = prov.get("extraction_tools", {})
        sources = prov.get("data_sources", {})
        for tool_name, pipeline in tools.items():
            src = sources.get(tool_name, [])
            print(f"  {tool_name}: {pipeline} ({', '.join(src) if src else 'N/A'})")

    print(f"\n{C}{'=' * 70}{X}\n")

print("render_clinical_output() defined")


## Load Track 1 Cases

12 RSNA CXR cases across 5 diagnostic categories. Each case includes clinical data
designed to test specific cross-modal contradiction rules when combined with real CXR model output.


In [ ]:
from benchmark_data.cases.registry import get_track1_cases
import copy, os
import benchmark_data.rsna

RSNA_IMG_DIR = os.path.join(os.path.dirname(benchmark_data.rsna.__file__), "images")

track1_cases = copy.deepcopy(get_track1_cases())
for case in track1_cases:
    img_name = os.path.basename(case["cxr"]["image_path"])
    case["cxr"]["image_path"] = os.path.join(RSNA_IMG_DIR, img_name)
    if case["cxr"].get("prior_image_path"):
        prior_name = os.path.basename(case["cxr"]["prior_image_path"])
        case["cxr"]["prior_image_path"] = os.path.join(RSNA_IMG_DIR, prior_name)

# Group by category
categories = {}
for case in track1_cases:
    cat = case["ground_truth"]["cxr_ground_truth"]["cxr_category"]
    categories.setdefault(cat, []).append(case)

print(f"Loaded {len(track1_cases)} Track 1 cases across {len(categories)} categories:")
for cat, cases in sorted(categories.items()):
    case_ids = [c["case_id"] for c in cases]
    print(f"  {cat}: {len(cases)} cases — {', '.join(case_ids)}")


## Run All 12 Cases

Each case runs the full 8-node pipeline with real CXR images. The CXR extraction module
makes 1-3 MedGemma GPU calls per image depending on findings detected and prior CXR availability.

Estimated time: ~25 minutes on A100 GPU (12 cases, ~2 min each).


In [ ]:
from cap_agent.agent.state import build_initial_state
import time

all_results = []
for i, case in enumerate(track1_cases):
    case_id = case["case_id"]
    print(f"[{i+1}/{len(track1_cases)}] Running {case_id}...")
    initial_state = build_initial_state(case)
    start = time.time()
    result = None
    for chunk in graph.stream(initial_state, stream_mode="values"):
        result = chunk
    elapsed = time.time() - start
    all_results.append({"case_id": case_id, "case": case, "result": result, "elapsed": elapsed})
    curb = result.get("curb65_score", {})
    cxr = result.get("cxr_analysis", {})
    consol = cxr.get("consolidation", {})
    print(f"  CURB65={curb.get('curb65')} ({curb.get('severity_tier')}) "
          f"Consolidation={'PRESENT' if consol.get('present') else 'absent'} "
          f"in {elapsed:.1f}s")

print(f"\nAll {len(all_results)} cases complete")
total_time = sum(r["elapsed"] for r in all_results)
print(f"Total pipeline time: {total_time:.1f}s ({total_time/len(all_results):.1f}s per case)")


## Category 1: Clear Pneumonia (3 cases)

These cases have unambiguous consolidation on CXR with concordant clinical findings
(focal crackles, bronchial breathing in the same zone). The pipeline should:
- Detect consolidation as PRESENT with high confidence
- Produce bounding boxes that overlap with RSNA ground truth annotations
- Report no cross-modal contradictions (findings are concordant)


In [ ]:
cat_cases = [r for r in all_results
             if r["case"]["ground_truth"]["cxr_ground_truth"]["cxr_category"] == "clear_pneumonia"]
print(f"Clear Pneumonia: {len(cat_cases)} cases\n")

for r in cat_cases:
    case = r["case"]
    result = r["result"]
    cxr = result.get("cxr_analysis", {})
    gt = case["ground_truth"]["cxr_ground_truth"]

    # Build predicted bboxes from CXR analysis
    pred_bboxes = []
    consol = cxr.get("consolidation", {})
    if consol.get("bounding_box"):
        pred_bboxes.append({"bbox": consol["bounding_box"], "label": "consolidation"})

    # Build GT bboxes
    gt_bboxes = [{"bbox": b, "label": "GT"} for b in gt.get("bboxes_normalized", [])]

    # Display image with overlays
    display_cxr_with_bbox(case["cxr"]["image_path"], pred_bboxes, gt_bboxes, r["case_id"])

    # Classification table
    print("  Classification:")
    for finding in ["consolidation", "pleural_effusion", "cardiomegaly", "edema", "atelectasis"]:
        f = cxr.get(finding, {})
        status = "PRESENT" if f.get("present") else "absent"
        print(f"    {finding}: {status} ({f.get('confidence', '?')})")

    # IoU calculation
    if pred_bboxes and gt_bboxes:
        best_iou = max(compute_iou(pred_bboxes[0]["bbox"], gb["bbox"]) for gb in gt_bboxes)
        print(f"  Best IoU: {best_iou:.3f}")
    else:
        print("  IoU: N/A (no predicted or GT bbox)")

    # Clinical output
    render_clinical_output(result, r["case_id"])


## Category 2: Bilateral Pneumonia (2 cases)

Bilateral consolidation with **unilateral** clinical signs (crackles in one zone only).
These cases are designed to trigger **CR-5** (laterality mismatch between CXR and exam findings).

The pipeline should:
- Detect consolidation in multiple regions
- Produce multiple bounding boxes matching RSNA ground truth
- Trigger CR-5 contradiction due to bilateral CXR vs unilateral crackles


In [ ]:
cat_cases = [r for r in all_results
             if r["case"]["ground_truth"]["cxr_ground_truth"]["cxr_category"] == "bilateral_pneumonia"]
print(f"Bilateral Pneumonia: {len(cat_cases)} cases\n")

for r in cat_cases:
    case = r["case"]
    result = r["result"]
    cxr = result.get("cxr_analysis", {})
    gt = case["ground_truth"]["cxr_ground_truth"]

    # Build predicted bboxes
    pred_bboxes = []
    consol = cxr.get("consolidation", {})
    if consol.get("bounding_box"):
        pred_bboxes.append({"bbox": consol["bounding_box"], "label": "consolidation"})

    # GT bboxes (multiple for bilateral)
    gt_bboxes = [{"bbox": b, "label": f"GT-{i+1}"} for i, b in enumerate(gt.get("bboxes_normalized", []))]

    display_cxr_with_bbox(case["cxr"]["image_path"], pred_bboxes, gt_bboxes, r["case_id"])

    # Classification
    print("  Classification:")
    for finding in ["consolidation", "pleural_effusion", "cardiomegaly", "edema", "atelectasis"]:
        f = cxr.get(finding, {})
        status = "PRESENT" if f.get("present") else "absent"
        print(f"    {finding}: {status} ({f.get('confidence', '?')})")

    # IoU (best match across multiple GT boxes)
    if pred_bboxes and gt_bboxes:
        best_iou = max(compute_iou(pred_bboxes[0]["bbox"], gb["bbox"]) for gb in gt_bboxes)
        print(f"  Best IoU: {best_iou:.3f}")

    # Check contradiction detection
    contradictions = result.get("contradictions_detected", [])
    cr_ids = [c["rule_id"] for c in contradictions]
    expected_crs = case["ground_truth"].get("contradictions", [])
    print(f"  Expected contradictions: {expected_crs}")
    print(f"  Detected contradictions: {cr_ids}")

    render_clinical_output(result, r["case_id"])


## Category 3: Subtle Pneumonia (2 cases)

Borderline CXR findings with small or faint opacities. These test the model's **sensitivity** —
can MedGemma detect subtle consolidation that is genuinely present?

The pipeline should:
- Detect consolidation (even at low confidence)
- Produce bounding boxes for the subtle findings
- Clinical findings (crackles, high CRP) are concordant, so no contradictions expected


In [ ]:
cat_cases = [r for r in all_results
             if r["case"]["ground_truth"]["cxr_ground_truth"]["cxr_category"] == "subtle_pneumonia"]
print(f"Subtle Pneumonia: {len(cat_cases)} cases\n")

for r in cat_cases:
    case = r["case"]
    result = r["result"]
    cxr = result.get("cxr_analysis", {})
    gt = case["ground_truth"]["cxr_ground_truth"]

    pred_bboxes = []
    consol = cxr.get("consolidation", {})
    if consol.get("bounding_box"):
        pred_bboxes.append({"bbox": consol["bounding_box"], "label": "consolidation"})

    gt_bboxes = [{"bbox": b, "label": "GT"} for b in gt.get("bboxes_normalized", [])]

    display_cxr_with_bbox(case["cxr"]["image_path"], pred_bboxes, gt_bboxes, r["case_id"])

    print("  Classification:")
    for finding in ["consolidation", "pleural_effusion", "cardiomegaly", "edema", "atelectasis"]:
        f = cxr.get(finding, {})
        status = "PRESENT" if f.get("present") else "absent"
        print(f"    {finding}: {status} ({f.get('confidence', '?')})")

    if pred_bboxes and gt_bboxes:
        best_iou = max(compute_iou(pred_bboxes[0]["bbox"], gb["bbox"]) for gb in gt_bboxes)
        print(f"  Best IoU: {best_iou:.3f}")
    else:
        detected = consol.get("present", False)
        print(f"  Consolidation detected: {detected} (GT: present)")
        if not detected:
            print(f"  NOTE: Subtle finding missed — sensitivity gap")

    render_clinical_output(result, r["case_id"])


## Category 4: Normal CXR (3 cases)

Normal chest X-rays paired with **abnormal clinical findings** (focal crackles, bronchial
breathing, elevated CRP). These test the pipeline's ability to detect **cross-modal
contradictions**:

- **CR-1:** Clinical exam suggests consolidation (crackles + bronchial breathing) but CXR is normal
- **CR-2:** CRP > 100 but CXR shows no consolidation (high CRP without radiological evidence)

The pipeline should correctly identify the CXR as normal (true negative) and flag the
discordance between clinical and radiological findings.


In [ ]:
cat_cases = [r for r in all_results
             if r["case"]["ground_truth"]["cxr_ground_truth"]["cxr_category"] == "normal"]
print(f"Normal CXR: {len(cat_cases)} cases\n")

for r in cat_cases:
    case = r["case"]
    result = r["result"]
    cxr = result.get("cxr_analysis", {})
    gt = case["ground_truth"]["cxr_ground_truth"]

    # Normal CXR should have no predicted bboxes
    pred_bboxes = []
    consol = cxr.get("consolidation", {})
    if consol.get("bounding_box"):
        pred_bboxes.append({"bbox": consol["bounding_box"], "label": "consolidation"})

    gt_bboxes = []  # No GT bboxes for normal CXR

    display_cxr_with_bbox(case["cxr"]["image_path"], pred_bboxes, gt_bboxes, r["case_id"])

    print("  Classification:")
    for finding in ["consolidation", "pleural_effusion", "cardiomegaly", "edema", "atelectasis"]:
        f = cxr.get(finding, {})
        status = "PRESENT" if f.get("present") else "absent"
        print(f"    {finding}: {status} ({f.get('confidence', '?')})")

    # True negative check
    consol_present = consol.get("present", False)
    gt_present = gt["consolidation_present"]
    tn = not consol_present and not gt_present
    print(f"  True negative: {'YES' if tn else 'NO (false positive)'}")

    # Contradiction detection
    contradictions = result.get("contradictions_detected", [])
    cr_ids = [c["rule_id"] for c in contradictions]
    expected_crs = case["ground_truth"].get("contradictions", [])
    print(f"  Expected contradictions: {expected_crs}")
    print(f"  Detected contradictions: {cr_ids}")
    for cr in expected_crs:
        if cr in cr_ids:
            print(f"    {cr}: DETECTED")
        else:
            print(f"    {cr}: MISSED")

    render_clinical_output(result, r["case_id"])


## Category 5: Abnormal Non-Pneumonia (2 cases)

CXR shows **pleural effusion** without consolidation, paired with clinical data that does
not suggest effusion (no crackles, no bronchial breathing). These test **CR-6** detection:

- **CR-6:** CXR shows unexpected abnormality (effusion) not accounted for by clinical assessment

The pipeline should detect effusion as an incidental finding and flag it for clinical review.


In [ ]:
cat_cases = [r for r in all_results
             if r["case"]["ground_truth"]["cxr_ground_truth"]["cxr_category"] == "abnormal_non_pneumonia"]
print(f"Abnormal Non-Pneumonia: {len(cat_cases)} cases\n")

for r in cat_cases:
    case = r["case"]
    result = r["result"]
    cxr = result.get("cxr_analysis", {})
    gt = case["ground_truth"]["cxr_ground_truth"]

    # No consolidation expected; look for effusion bbox
    pred_bboxes = []
    effusion = cxr.get("pleural_effusion", {})
    if effusion.get("bounding_box"):
        pred_bboxes.append({"bbox": effusion["bounding_box"], "label": "effusion"})
    consol = cxr.get("consolidation", {})
    if consol.get("bounding_box"):
        pred_bboxes.append({"bbox": consol["bounding_box"], "label": "consolidation"})

    gt_bboxes = []  # No consolidation GT for these cases

    display_cxr_with_bbox(case["cxr"]["image_path"], pred_bboxes, gt_bboxes, r["case_id"])

    print("  Classification:")
    for finding in ["consolidation", "pleural_effusion", "cardiomegaly", "edema", "atelectasis"]:
        f = cxr.get(finding, {})
        status = "PRESENT" if f.get("present") else "absent"
        print(f"    {finding}: {status} ({f.get('confidence', '?')})")

    # Effusion detection check
    effusion_detected = effusion.get("present", False)
    print(f"  Effusion detected: {effusion_detected}")

    # Contradiction detection
    contradictions = result.get("contradictions_detected", [])
    cr_ids = [c["rule_id"] for c in contradictions]
    expected_crs = case["ground_truth"].get("contradictions", [])
    print(f"  Expected contradictions: {expected_crs}")
    print(f"  Detected contradictions: {cr_ids}")

    render_clinical_output(result, r["case_id"])


## Summary: Classification Accuracy

Confusion matrix for consolidation detection across all 12 cases.


In [ ]:
import plotly.graph_objects as go

tp = fp = tn = fn = 0
for r in all_results:
    gt_present = r["case"]["ground_truth"]["cxr_ground_truth"]["consolidation_present"]
    pred_present = r["result"].get("cxr_analysis", {}).get("consolidation", {}).get("present", False)
    if gt_present and pred_present:
        tp += 1
    elif gt_present and not pred_present:
        fn += 1
    elif not gt_present and pred_present:
        fp += 1
    else:
        tn += 1

accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

fig = go.Figure(data=go.Heatmap(
    z=[[tp, fn], [fp, tn]],
    x=["Predicted +", "Predicted -"],
    y=["Actual +", "Actual -"],
    text=[[str(tp), str(fn)], [str(fp), str(tn)]],
    texttemplate="%{text}",
    colorscale="Blues",
))
fig.update_layout(
    title=f"Consolidation Detection: {tp+tn}/{tp+tn+fp+fn} correct ({accuracy:.0%} accuracy)",
    width=500, height=400,
)
fig.show()

print(f"\nConsolidation Detection Metrics:")
print(f"  True Positives:  {tp}")
print(f"  True Negatives:  {tn}")
print(f"  False Positives: {fp}")
print(f"  False Negatives: {fn}")
print(f"  Accuracy:        {accuracy:.2%}")
print(f"  Sensitivity:     {sensitivity:.2%}")
print(f"  Specificity:     {specificity:.2%}")


## Summary: Localization IoU

Bounding box IoU (Intersection-over-Union) per case for cases with ground truth annotations.
IoU > 0.5 is typically considered a good localization; IoU > 0.2 is acceptable.


In [ ]:
import plotly.graph_objects as go

iou_cases = []
iou_values = []
iou_colors = []

for r in all_results:
    gt = r["case"]["ground_truth"]["cxr_ground_truth"]
    gt_bboxes = gt.get("bboxes_normalized", [])
    if not gt_bboxes:
        continue  # Skip cases without GT bboxes

    cxr = r["result"].get("cxr_analysis", {})
    consol = cxr.get("consolidation", {})
    pred_bbox = consol.get("bounding_box")

    if pred_bbox and len(pred_bbox) == 4:
        best_iou = max(compute_iou(pred_bbox, gb) for gb in gt_bboxes)
    else:
        best_iou = 0.0

    iou_cases.append(r["case_id"])
    iou_values.append(best_iou)
    if best_iou >= 0.5:
        iou_colors.append("#4CAF50")  # Green
    elif best_iou >= 0.2:
        iou_colors.append("#FF9800")  # Orange
    else:
        iou_colors.append("#F44336")  # Red

if iou_cases:
    fig = go.Figure(data=go.Bar(
        x=iou_values,
        y=iou_cases,
        orientation="h",
        marker_color=iou_colors,
        text=[f"{v:.3f}" for v in iou_values],
        textposition="auto",
    ))
    fig.add_vline(x=0.5, line_dash="dash", line_color="green", annotation_text="Good (0.5)")
    fig.add_vline(x=0.2, line_dash="dash", line_color="orange", annotation_text="Acceptable (0.2)")
    fig.update_layout(
        title="Localization IoU per Case",
        xaxis_title="IoU",
        xaxis=dict(range=[0, 1]),
        width=600, height=max(250, 50 * len(iou_cases)),
    )
    fig.show()

    mean_iou = sum(iou_values) / len(iou_values) if iou_values else 0
    good = sum(1 for v in iou_values if v >= 0.5)
    acceptable = sum(1 for v in iou_values if 0.2 <= v < 0.5)
    poor = sum(1 for v in iou_values if v < 0.2)
    print(f"\nLocalization Summary ({len(iou_values)} cases with GT bboxes):")
    print(f"  Mean IoU:    {mean_iou:.3f}")
    print(f"  Good (>=0.5):       {good}")
    print(f"  Acceptable (>=0.2): {acceptable}")
    print(f"  Poor (<0.2):        {poor}")
else:
    print("No cases with ground truth bounding boxes")


## Verification Checklist

Per-case pass/fail for classification accuracy and localization quality.


In [ ]:
ESC = chr(27)
B = f"{ESC}[1m"; G = f"{ESC}[92m"; R = f"{ESC}[91m"; Y = f"{ESC}[93m"
C = f"{ESC}[96m"; X = f"{ESC}[0m"

print(f"\n{B}{C}{'=' * 70}")
print(f"  CXR IMAGING ANALYSIS — VERIFICATION")
print(f"{'=' * 70}{X}\n")

header = f"  {'Case':<20} {'Classification':>15} {'IoU':>8} {'Contradictions':>16} {'Status':>8}"
print(header)
print(f"  {'─' * 67}")

total_pass = 0
total_fail = 0

for r in all_results:
    case_id = r["case_id"]
    result = r["result"]
    case = r["case"]
    gt = case["ground_truth"]["cxr_ground_truth"]
    cxr = result.get("cxr_analysis", {})

    # Classification check
    gt_present = gt["consolidation_present"]
    pred_present = cxr.get("consolidation", {}).get("present", False)
    class_correct = gt_present == pred_present
    class_str = f"{G}CORRECT{X}" if class_correct else f"{R}WRONG{X}"

    # IoU check (only for cases with GT bboxes)
    gt_bboxes = gt.get("bboxes_normalized", [])
    pred_bbox = cxr.get("consolidation", {}).get("bounding_box")
    if gt_bboxes and pred_bbox and len(pred_bbox) == 4:
        best_iou = max(compute_iou(pred_bbox, gb) for gb in gt_bboxes)
        iou_str = f"{best_iou:.3f}"
    elif gt_bboxes and not pred_bbox:
        best_iou = 0.0
        iou_str = f"{R}0.000{X}"
    else:
        best_iou = None
        iou_str = "N/A"

    # Contradiction check
    expected_crs = set(case["ground_truth"].get("contradictions", []))
    actual_crs = {c["rule_id"] for c in result.get("contradictions_detected", [])}
    if not expected_crs:
        cr_str = f"{G}none{X}"
        cr_ok = True
    else:
        found = expected_crs & actual_crs
        cr_str = f"{len(found)}/{len(expected_crs)}"
        cr_ok = found == expected_crs

    # Overall status
    iou_ok = best_iou is None or best_iou >= 0.1
    all_ok = class_correct and cr_ok and iou_ok
    if all_ok:
        status = f"{G}PASS{X}"
        total_pass += 1
    else:
        status = f"{R}FAIL{X}"
        total_fail += 1

    print(f"  {case_id:<20} {class_str:>24} {iou_str:>17} {cr_str:>25} {status:>17}")

print(f"\n{B}Results: {total_pass}/{total_pass + total_fail} cases passed{X}")
if total_fail == 0:
    print(f"{G}All CXR imaging analysis checks passed!{X}")
else:
    print(f"{Y}{total_fail} case(s) need review{X}")

print(f"\n{C}{'=' * 70}{X}")
